# TP Data Storytelling – Accidents corporels de la route 2024 (BAAC)
## Option C : la route et ses conditions

**Binôme :** _Nom 1_ – _Nom 2_  ·  M1 Cybersécurité et Sciences des Données, Université Paris 8  ·  2026/2027

**Question directrice (reformulée) :** sur quels types de routes, à quelles vitesses autorisées et dans quelles conditions (météo, état de la chaussée, luminosité) les accidents corporels de 2024 sont-ils les plus graves ? Et où le gestionnaire de voirie doit-il agir en priorité ?

**Public cible :** gestionnaire de voirie (département, métropole).

**Objectif de ce notebook :** préparer deux fichiers propres pour Tableau, chacun avec un grain clair :
- `usagers_enrichis.csv` : **une ligne par usager**, avec son véhicule, l'accident et le lieu ;
- `accidents.csv` : **une ligne par accident**, avec le nombre de véhicules, d'usagers, de tués, d'hospitalisés et de blessés légers.

**Étapes :** 0. Préparation · 1. Chargement et inspection · 2. Contrôles qualité · 3. Nettoyage et recodage · 4. Variables dérivées · 5. Jointures · 6. Vérifications et export · 7. Tableau des contrôles · 8. Chiffres clés pour la story

Source : ONISR, base BAAC 2024 (data.gouv.fr). Les 4 CSV d'origine doivent être placés dans `data/raw/`.

## 0. Préparation
On importe les bibliothèques et on définit les chemins du projet. Le notebook est rangé dans `notebooks/`, donc la racine du projet est le dossier parent. Les fichiers bruts sont lus dans `data/raw/` et les sorties sont écrites dans `data/processed/` et `reports/`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
REPORTS = ROOT / "reports"
PROCESSED.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

ANNEE = 2024
print("Racine du projet :", ROOT)
print("Fichiers bruts   :", [f.name for f in sorted(RAW.glob("*.csv"))])

Racine du projet : C:\Users\akils\Downloads\road-safety-data-analysis-2024\road-safety-data-analysis-2024
Fichiers bruts   : ['caracteristiques-2024.csv', 'lieux-2024.csv', 'usagers-2024.csv', 'vehicules-2024.csv']


## 1. Charger et inspecter les 4 tables
Le sujet demande de **vérifier soi-même le séparateur et l'encodage**. La fonction `lire()` s'en charge :
- elle essaie d'abord l'UTF-8, puis passe au Latin-1 si la lecture échoue ;
- elle compte les `;` et les `,` de la ligne d'en-tête pour trouver le séparateur.

**Toutes les colonnes sont lues en texte** (`dtype=str`). Les identifiants (`Num_Acc`, `id_vehicule`, `id_usager`) et les codes `dep`/`com` gardent ainsi leur forme exacte, par exemple `2A` ou `01`. Les autres colonnes sont converties plus tard, une par une et de façon contrôlée. Avec `keep_default_na=False`, pandas ne transforme rien en NaN en silence : c'est nous qui décidons, à l'étape 3, ce qui compte comme « non renseigné ».

In [2]:
def trouver_fichier(*motifs):
    for motif in motifs:
        fichiers = sorted(RAW.glob(f"*{motif}*{ANNEE}*.csv"))
        if fichiers:
            return fichiers[0]
    raise FileNotFoundError(f"Aucun fichier {motifs} pour {ANNEE} dans {RAW}")

def lire(*motifs):
    chemin = trouver_fichier(*motifs)
    brut = chemin.read_bytes()
    try:
        brut.decode("utf-8-sig")
        enc = "utf-8-sig"
    except UnicodeDecodeError:
        enc = "latin-1"
    entete = brut.split(b"\n", 1)[0].decode(enc)
    sep = ";" if entete.count(";") >= entete.count(",") else ","
    df = pd.read_csv(chemin, sep=sep, encoding=enc, dtype=str, keep_default_na=False)
    df.columns = df.columns.str.strip().str.replace('"', "", regex=False)
    df = df.apply(lambda s: s.str.strip())          # espaces parasites (ex. " -1")
    # Noms de colonnes harmonisés (ils varient selon les années : Accident_Id, an_nais…)
    df = df.rename(columns={"Accident_Id": "Num_Acc", "an_nais": "An_nais"})
    print(f"{chemin.name:<32} séparateur '{sep}'  encodage {enc:<9} {df.shape[0]:>8,} lignes × {df.shape[1]} colonnes")
    return df

carac = lire("carac", "carc", "caract")
lieux = lire("lieux")
vehicules = lire("vehicules")
usagers = lire("usagers")

caracteristiques-2024.csv        séparateur ';'  encodage utf-8-sig   54,402 lignes × 15 colonnes
lieux-2024.csv                   séparateur ';'  encodage utf-8-sig   70,248 lignes × 18 colonnes
vehicules-2024.csv               séparateur ';'  encodage utf-8-sig   92,678 lignes × 11 colonnes
usagers-2024.csv                 séparateur ';'  encodage utf-8-sig  125,187 lignes × 16 colonnes


In [3]:
for nom, df in [("CARACTERISTIQUES", carac), ("LIEUX", lieux), ("VEHICULES", vehicules), ("USAGERS", usagers)]:
    print(f"\n===== {nom} : {df.shape[0]:,} lignes × {df.shape[1]} colonnes =====")
    print(df.dtypes.to_string())
    display(df.head(3))


===== CARACTERISTIQUES : 54,402 lignes × 15 colonnes =====
Num_Acc    str
jour       str
mois       str
an         str
hrmn       str
lum        str
dep        str
com        str
agg        str
int        str
atm        str
col        str
adr        str
lat        str
long       str


,Num_Acc,jour,mois,an,hrmn,lum,dep,com,agg,int,atm,col,adr,lat,long
0,202400000001,25,03,2024,07:40,2,70,70285,1,1,5,1,D438,"47,56277000","6,75832000"
1,202400000002,20,03,2024,15:05,1,21,21054,2,3,7,6,HOTEL DIEU (RUE DE L'),"47,02109000","4,83755000"
2,202400000003,22,03,2024,19:30,2,15,15012,1,1,1,6,Allée des Tilleuls,"44,90238400","2,49641800"



===== LIEUX : 70,248 lignes × 18 colonnes =====
Num_Acc    str
catr       str
voie       str
v1         str
v2         str
circ       str
nbv        str
vosp       str
prof       str
pr         str
pr1        str
plan       str
lartpc     str
larrout    str
surf       str
infra      str
situ       str
vma        str


,Num_Acc,catr,voie,v1,v2,circ,nbv,vosp,prof,pr,pr1,plan,lartpc,larrout,surf,infra,situ,vma
0,202400000001,3,D438,0,N/A,2,2,0,1,1,260,2,,7,1,0,1,90
1,202400000002,4,HOTEL DIEU (RUE DE L'),0,N/A,2,2,0,1,-1,-1,1,,-1,9,0,1,30
2,202400000002,4,POTERNE (RUE),0,N/A,1,1,0,1,-1,-1,1,,-1,9,0,1,30



===== VEHICULES : 92,678 lignes × 11 colonnes =====
Num_Acc        str
id_vehicule    str
num_veh        str
senc           str
catv           str
obs            str
obsm           str
choc           str
manv           str
motor          str
occutc         str


,Num_Acc,id_vehicule,num_veh,senc,catv,obs,obsm,choc,manv,motor,occutc
0,202400000001,155 781 758,A01,1,7,0,2,1,13,1,
1,202400000001,155 781 759,B01,2,14,0,2,2,21,1,
2,202400000002,155 781 757,A01,1,10,0,1,3,15,1,



===== USAGERS : 125,187 lignes × 16 colonnes =====
Num_Acc        str
id_usager      str
id_vehicule    str
num_veh        str
place          str
catu           str
grav           str
sexe           str
An_nais        str
trajet         str
secu1          str
secu2          str
secu3          str
locp           str
actp           str
etatp          str


,Num_Acc,id_usager,id_vehicule,num_veh,place,catu,grav,sexe,An_nais,trajet,secu1,secu2,secu3,locp,actp,etatp
0,202400000001,203 988 581,155 781 758,A01,1,1,3,1,2003,2,1,-1,-1,-1,-1,-1
1,202400000001,203 988 582,155 781 759,B01,1,1,1,1,1997,4,1,-1,-1,-1,-1,-1
2,202400000002,203 988 579,155 781 757,A01,10,3,3,2,1927,5,0,-1,-1,3,3,1


## 2. Contrôler la qualité
Avant de transformer quoi que ce soit, on documente l'état des données brutes. Chaque contrôle est ajouté dans une liste `controles`, qui sert à produire le **tableau des contrôles** demandé dans les livrables (section 7).

Selon la variable, une valeur `-1`, une case vide, un `0` ou un `.` signifie « non renseigné » ou « sans objet ». On compte d'abord ces valeurs colonne par colonne.

In [4]:
controles = []
def controle(nom, resultat, ok, commentaire=""):
    controles.append({"Contrôle": nom, "Résultat": resultat,
                      "Statut": "OK" if ok else "À signaler", "Commentaire / action": commentaire})
    print(f"[{'OK' if ok else '!!'}] {nom} : {resultat}")

### 2.1 Unicité de `Num_Acc` dans CARACTERISTIQUES et LIEUX

In [5]:
dup_carac = carac["Num_Acc"].duplicated().sum()
controle("Num_Acc unique dans CARACTERISTIQUES", f"{dup_carac} doublon(s) sur {len(carac):,} lignes", dup_carac == 0,
         "Clé primaire de la table accident")

dup_lieux = lieux["Num_Acc"].duplicated().sum()
acc_multi_lieux = lieux.loc[lieux["Num_Acc"].duplicated(keep=False), "Num_Acc"].nunique()
controle("Num_Acc unique dans LIEUX", f"{dup_lieux:,} ligne(s) en double ; {acc_multi_lieux:,} accident(s) décrits sur plusieurs voies",
         dup_lieux == 0, "Accident au croisement de plusieurs voies : on garde la 1re ligne (lieu principal) pour joindre en 1-1")
if dup_lieux:
    display(lieux[lieux["Num_Acc"].duplicated(keep=False)].sort_values("Num_Acc").head(10))

sans_lieu = (~carac["Num_Acc"].isin(lieux["Num_Acc"])).sum()
lieu_orphelin = (~lieux["Num_Acc"].isin(carac["Num_Acc"])).sum()
controle("Correspondance CARACTERISTIQUES ↔ LIEUX",
         f"{sans_lieu} accident(s) sans lieu, {lieu_orphelin} lieu(x) sans accident",
         sans_lieu == 0 and lieu_orphelin == 0)

NameError: name 'controle' is not defined

### 2.2 Valeurs manquantes et fréquence des `-1`, par colonne

In [6]:
def profil_non_renseignes(df, nom):
    res = pd.DataFrame({
        "vide": (df == "").sum(),
        "valeur_-1": (df == "-1").sum(),
        "point": (df == ".").sum(),
    })
    res["total"] = res.sum(axis=1)
    res["pct_lignes"] = (100 * res["total"] / len(df)).round(2)
    res.insert(0, "table", nom)
    return res[res["total"] > 0].sort_values("pct_lignes", ascending=False)

manquants = pd.concat([profil_non_renseignes(carac, "caracteristiques"), profil_non_renseignes(lieux, "lieux"),
                       profil_non_renseignes(vehicules, "vehicules"), profil_non_renseignes(usagers, "usagers")])
manquants.to_csv(REPORTS / "valeurs_non_renseignees.csv", encoding="utf-8")
display(manquants)

nb_moins1 = int(manquants["valeur_-1"].sum())
controle("Valeurs -1 (non renseigné), toutes tables", f"{nb_moins1:,} cellules ; détail dans reports/valeurs_non_renseignees.csv",
         True, "Transformées en NaN à l'étape 3 ; poids indiqué dans les limites de la story")

,table,vide,valeur_-1,point,total,pct_lignes
col,caracteristiques,0,6,0,6,0.01
adr,caracteristiques,2,0,0,2,0.00
lartpc,lieux,70215,0,0,70215,99.95
larrout,lieux,0,48646,0,48646,69.25
pr1,lieux,0,27432,0,27432,39.05
pr,lieux,0,27364,0,27364,38.95
v1,lieux,0,16272,0,16272,23.16
circ,lieux,0,4354,0,4354,6.20
nbv,lieux,0,4178,0,4178,5.95
vosp,lieux,0,3832,0,3832,5.45


NameError: name 'controle' is not defined

Pour l'option C, on regarde de plus près les variables du lieu et des conditions. Remarque : les colonnes `voie`, `v1`, `v2`, `pr`, `pr1`, `lartpc` et `larrout` sont très souvent vides, mais nous ne les utilisons pas.

In [7]:
vars_c = ["catr", "vma", "circ", "nbv", "prof", "plan", "surf", "infra", "lum", "agg", "int", "atm", "col"]
display(manquants[manquants.index.isin(vars_c)])

,table,vide,valeur_-1,point,total,pct_lignes
col,caracteristiques,0,6,0,6,0.01
circ,lieux,0,4354,0,4354,6.20
nbv,lieux,0,4178,0,4178,5.95
vma,lieux,0,3630,0,3630,5.17
infra,lieux,0,812,0,812,1.16
prof,lieux,0,50,0,50,0.07
plan,lieux,0,40,0,40,0.06
surf,lieux,0,38,0,38,0.05


### 2.3 Années de naissance aberrantes

In [8]:
an_nais = pd.to_numeric(usagers["An_nais"], errors="coerce")
nb_absent = an_nais.isna().sum()
aberrant = (an_nais < 1900) | (an_nais > ANNEE) | ((ANNEE - an_nais) > 105)
controle("Années de naissance aberrantes (avant 1900, après 2024, ou âge > 105 ans)",
         f"{aberrant.sum()} aberrante(s), {nb_absent:,} absente(s) sur {len(usagers):,} usagers",
         aberrant.sum() == 0, "Âge mis à NaN pour ces usagers")
print(an_nais.describe())

NameError: name 'controle' is not defined

### 2.4 Coordonnées nulles, absentes ou hors de France
`lat` et `long` sont parfois écrits avec une virgule décimale : on remplace la virgule par un point avant de convertir. Pour la France métropolitaine, on vérifie que le point tombe dans un rectangle qui englobe le territoire, Corse comprise (latitude 41 à 51,5 ; longitude −5,5 à 10).

In [9]:
def en_nombre(s):
    return pd.to_numeric(s.astype(str).str.replace(",", ".", regex=False), errors="coerce")

lat, lon = en_nombre(carac["lat"]), en_nombre(carac["long"])
outremer = carac["dep"].str.startswith("97") | carac["dep"].isin(["986", "987", "988"])
coord_absentes = lat.isna() | lon.isna()
coord_nulles = (lat == 0) & (lon == 0)
hors_metro = ~outremer & ~coord_absentes & ~coord_nulles & ~(lat.between(41, 51.5) & lon.between(-5.5, 10))

controle("Coordonnées absentes ou nulles (0,0)", f"{coord_absentes.sum()} absentes, {coord_nulles.sum()} nulles",
         (coord_absentes | coord_nulles).sum() == 0, "Gardées dans les comptages mais exclues de la carte")
controle("Coordonnées hors de France métropolitaine (départements métropolitains)", f"{hors_metro.sum()} accident(s)",
         hors_metro.sum() == 0, "Signalées ; exclues de la carte")
print("Accidents en outre-mer :", outremer.sum())

[OK] Coordonnées absentes ou nulles (0,0) : 0 absentes, 0 nulles
[!!] Coordonnées hors de France métropolitaine (départements métropolitains) : 3 accident(s)
Accidents en outre-mer : 3344


### 2.5 Véhicules sans usager (le sujet en annonce 24 en 2024)

In [10]:
cle_veh = vehicules["Num_Acc"] + "|" + vehicules["id_vehicule"]
cle_usa = usagers["Num_Acc"] + "|" + usagers["id_vehicule"]
veh_sans_usager = (~cle_veh.isin(cle_usa)).sum()
controle("id_vehicule sans aucun usager", f"{veh_sans_usager} véhicule(s)", True,
         "Conforme à la documentation ONISR (usagers en fuite) ; comptés dans nb_vehicules")

usager_sans_veh = (~cle_usa.isin(cle_veh)).sum()
controle("Usagers rattachés à un véhicule absent de VEHICULES", f"{usager_sans_veh} usager(s)", usager_sans_veh == 0)

dup_veh = cle_veh.duplicated().sum()
controle("(Num_Acc, id_vehicule) unique dans VEHICULES", f"{dup_veh} doublon(s)", dup_veh == 0)
dup_usa = usagers["id_usager"].duplicated().sum()
controle("id_usager unique dans USAGERS", f"{dup_usa} doublon(s)", dup_usa == 0)

[OK] id_vehicule sans aucun usager : 24 véhicule(s)
[OK] Usagers rattachés à un véhicule absent de VEHICULES : 0 usager(s)
[OK] (Num_Acc, id_vehicule) unique dans VEHICULES : 0 doublon(s)


[OK] id_usager unique dans USAGERS : 0 doublon(s)


### 2.6 Vitesse maximale autorisée (`vma`) et gravité (`grav`)

In [9]:
VMA_VALIDES = [10, 20, 30, 40, 45, 50, 60, 70, 80, 90, 100, 110, 130]
vma = pd.to_numeric(lieux["vma"], errors="coerce")
print(vma.value_counts(dropna=False).sort_index().to_string())
vma_hors = (vma.notna() & (vma != -1) & ~vma.isin(VMA_VALIDES)).sum()
controle("Vitesse maximale autorisée hors des valeurs réglementaires", f"{vma_hors} lieu(x) (ex. 0, 500…)",
         vma_hors == 0, "Erreurs de saisie probables : mises à NaN")

grav_nr = (~usagers["grav"].isin(["1", "2", "3", "4"])).sum()
controle("Gravité non renseignée (grav)", f"{grav_nr} usager(s)", grav_nr == 0,
         "Si présents : usagers en fuite le plus souvent ; ni tués ni blessés dans les comptages")

vma
-1       3630
 0          1
 1         21
 2         12
 3          8
 4          1
 5         43
 6          9
 10       172
 15        73
 16         1
 20       301
 25        67
 30     13612
 35         2
 40        96
 45        21
 50     31560
 55         1
 60       124
 70      4177
 75         1
 80      8221
 85         1
 90      5247
 95         1
 100       16
 110     1791
 130     1009
 140        1
 300        2
 301        1
 500       21
 700        1
 800        1
 900        2


NameError: name 'controle' is not defined

## 3. Nettoyer et recoder
1. Chaque variable codée devient un **nombre** et les codes « non renseigné » deviennent des **NaN explicites**. On utilise pour cela la nomenclature de la *Description des bases de données annuelles* (ONISR).
2. On remplace les codes par des **libellés lisibles**. Un code absent du dictionnaire devient lui aussi NaN, et on affiche combien de valeurs sont concernées.
3. La date et l'heure deviennent de **vrais types** (`datetime`), et `lat`/`long` deviennent des **décimaux**.

Dans Tableau, les NaN apparaîtront comme « Non renseigné ». Ils ne sont pas supprimés : on doit pouvoir indiquer leur poids.

In [10]:
def en_code(s, manquants=(-1,)):
    x = pd.to_numeric(s.replace({"": np.nan, ".": np.nan}), errors="coerce")
    return x.mask(x.isin(manquants)).astype("Int64")

def libelle(s, dico, nom):
    lab = s.map(dico)
    hors = s.notna() & lab.isna()
    if hors.sum():
        print(f"  {nom}: {hors.sum()} valeur(s) hors nomenclature -> NaN : {sorted(s[hors].unique().tolist())}")
    return lab

NR = "Non renseigné"
LUM = {1: "Plein jour", 2: "Crépuscule ou aube", 3: "Nuit sans éclairage public",
       4: "Nuit avec éclairage non allumé", 5: "Nuit avec éclairage allumé"}
LUM_GROUPE = {1: "Jour", 2: "Aube / crépuscule", 3: "Nuit non éclairée", 4: "Nuit non éclairée", 5: "Nuit éclairée"}
AGG = {1: "Hors agglomération", 2: "En agglomération"}
INT = {1: "Hors intersection", 2: "Intersection en X", 3: "Intersection en T", 4: "Intersection en Y",
       5: "Intersection à plus de 4 branches", 6: "Giratoire", 7: "Place", 8: "Passage à niveau", 9: "Autre intersection"}
ATM = {1: "Normale", 2: "Pluie légère", 3: "Pluie forte", 4: "Neige - grêle", 5: "Brouillard - fumée",
       6: "Vent fort - tempête", 7: "Temps éblouissant", 8: "Temps couvert", 9: "Autre"}
ATM_GROUPE = {1: "Normale", 2: "Pluie", 3: "Pluie", 4: "Neige / grêle", 5: "Brouillard", 6: "Vent fort",
              7: "Éblouissant", 8: "Couvert", 9: "Autre"}
COL = {1: "Frontale (2 véhicules)", 2: "Par l'arrière (2 véhicules)", 3: "Par le côté (2 véhicules)",
       4: "En chaîne (3 véhicules et +)", 5: "Collisions multiples (3 véhicules et +)", 6: "Autre collision",
       7: "Sans collision"}
CATR = {1: "Autoroute", 2: "Route nationale", 3: "Route départementale", 4: "Voie communale",
        5: "Hors réseau public", 6: "Parc de stationnement", 7: "Route de métropole urbaine", 9: "Autre"}
CIRC = {1: "Sens unique", 2: "Bidirectionnelle", 3: "Chaussées séparées", 4: "Voies d'affectation variable"}
PROF = {1: "Plat", 2: "Pente", 3: "Sommet de côte", 4: "Bas de côte"}
PLAN = {1: "Ligne droite", 2: "Courbe à gauche", 3: "Courbe à droite", 4: "En S"}
PLAN_GROUPE = {1: "Ligne droite", 2: "Virage", 3: "Virage", 4: "Virage"}
SURF = {1: "Normale", 2: "Mouillée", 3: "Flaques", 4: "Inondée", 5: "Enneigée", 6: "Boue", 7: "Verglacée",
        8: "Corps gras - huile", 9: "Autre"}
SURF_GROUPE = {1: "Sèche (normale)", 2: "Mouillée", 3: "Mouillée", 4: "Mouillée", 5: "Neige / verglas",
               7: "Neige / verglas", 6: "Autre", 8: "Autre", 9: "Autre"}
INFRA = {0: "Aucune", 1: "Souterrain - tunnel", 2: "Pont - autopont", 3: "Bretelle d'échangeur",
         4: "Voie ferrée", 5: "Carrefour aménagé", 6: "Zone piétonne", 7: "Zone de péage", 8: "Chantier", 9: "Autre"}
GRAV = {1: "Indemne", 2: "Tué", 3: "Blessé hospitalisé", 4: "Blessé léger"}
CATU = {1: "Conducteur", 2: "Passager", 3: "Piéton"}
SEXE = {1: "Homme", 2: "Femme"}
CATV_GROUPE = {
    **{c: "Vélo" for c in [1, 80]},
    **{c: "Trottinette / EDP" for c in [50, 60]},
    **{c: "Deux-roues motorisé" for c in [2, 30, 31, 32, 33, 34, 41, 42, 43]},
    **{c: "Voiture" for c in [7]},
    **{c: "Utilitaire" for c in [10]},
    **{c: "Poids lourd" for c in [13, 14, 15, 16, 17]},
    **{c: "Transport en commun" for c in [37, 38, 39, 40]},
    **{c: "Autre" for c in [3, 20, 21, 35, 36, 99]},
}

### 3.1 Caractéristiques : codes, date et heure, coordonnées

In [13]:
c = carac.copy()
for col in ["jour", "mois", "an", "lum", "agg", "int", "atm", "col"]:
    c[col] = en_code(c[col], manquants=(-1, 0))

c["date"] = pd.to_datetime(dict(year=c["an"], month=c["mois"], day=c["jour"]), errors="coerce")
hm = c["hrmn"].str.replace(":", "", regex=False).str.zfill(4)
c["heure"] = pd.to_numeric(hm.str[:2], errors="coerce").astype("Int64")
c["minute"] = pd.to_numeric(hm.str[2:], errors="coerce").astype("Int64")
c["date_heure"] = c["date"] + pd.to_timedelta(c["heure"].astype("float"), unit="h") \
                            + pd.to_timedelta(c["minute"].astype("float"), unit="m")

c["lat"], c["long"] = en_nombre(c["lat"]), en_nombre(c["long"])
c.loc[(c["lat"] == 0) & (c["long"] == 0), ["lat", "long"]] = np.nan
c["zone"] = np.where(c["dep"].str.startswith("97") | c["dep"].isin(["986", "987", "988"]), "Outre-mer", "Métropole")
c["coord_valides"] = (c["lat"].notna() & c["long"].notna() &
                      ((c["zone"] == "Outre-mer") | (c["lat"].between(41, 51.5) & c["long"].between(-5.5, 10)))).astype(int)

c["luminosite"] = libelle(c["lum"], LUM, "lum")
c["luminosite_groupe"] = libelle(c["lum"], LUM_GROUPE, "lum")
c["agglomeration"] = libelle(c["agg"], AGG, "agg")
c["intersection"] = libelle(c["int"], INT, "int")
c["meteo"] = libelle(c["atm"], ATM, "atm")
c["meteo_groupe"] = libelle(c["atm"], ATM_GROUPE, "atm")
c["type_collision"] = libelle(c["col"], COL, "col")

print("Dates non valides :", c["date"].isna().sum(), "| heures non valides :", c["heure"].isna().sum())
print(c[["date_heure", "lat", "long"]].dtypes)

Dates non valides : 0 | heures non valides : 0
date_heure    datetime64[us]
lat                  float64
long                 float64
dtype: object


### 3.2 Lieux : codes et vitesse maximale autorisée
En 2024, LIEUX contient plusieurs lignes pour certains accidents : un accident à une intersection est décrit sur chacune des voies concernées. Le sujet définit LIEUX comme « le lieu principal d'un accident ». On garde donc **la première ligne de chaque accident**, ce qui permet une jointure 1-1 avec CARACTERISTIQUES. Ce choix est signalé dans les limites. Une `vma` hors des valeurs réglementaires devient NaN. On crée aussi des **classes de vitesse** plus lisibles dans Tableau.

In [14]:
l = lieux.drop_duplicates(subset="Num_Acc", keep="first").copy()
print("Lignes LIEUX supprimées (doublons) :", len(lieux) - len(l))

for col in ["catr", "circ", "prof", "plan", "surf", "infra"]:
    l[col] = en_code(l[col], manquants=(-1,))
l["nbv"] = en_code(l["nbv"], manquants=(-1,))         # "#ERREUR" -> NaN
l["vma"] = en_code(l["vma"], manquants=(-1,))
l.loc[~l["vma"].isin(VMA_VALIDES), "vma"] = pd.NA

def classe_vitesse(v):
    if pd.isna(v): return np.nan
    if v <= 30: return "1. 30 km/h ou moins"
    if v <= 50: return "2. 40-50 km/h"
    if v <= 70: return "3. 60-70 km/h"
    if v <= 80: return "4. 80 km/h"
    if v <= 100: return "5. 90-100 km/h"
    if v <= 110: return "6. 110 km/h"
    return "7. 130 km/h"
l["classe_vitesse"] = l["vma"].map(classe_vitesse)

l["categorie_route"] = libelle(l["catr"], CATR, "catr")
l["regime_circulation"] = libelle(l["circ"], CIRC, "circ")
l["profil_route"] = libelle(l["prof"], PROF, "prof")
l["trace_route"] = libelle(l["plan"], PLAN, "plan")
l["trace_groupe"] = libelle(l["plan"], PLAN_GROUPE, "plan")
l["etat_surface"] = libelle(l["surf"], SURF, "surf")
l["surface_groupe"] = libelle(l["surf"], SURF_GROUPE, "surf")
l["infrastructure"] = libelle(l["infra"], INFRA, "infra")
display(l["classe_vitesse"].value_counts(dropna=False).sort_index())

Lignes LIEUX supprimées (doublons) : 15846


classe_vitesse
1. 30 km/h ou moins     9510
2. 40-50 km/h          24110
3. 60-70 km/h           3966
4. 80 km/h              7176
5. 90-100 km/h          5057
6. 110 km/h             1787
7. 130 km/h             1007
NaN                     1789
Name: count, dtype: int64

### 3.3 Véhicules et usagers

In [13]:
v = vehicules.copy()
v["catv"] = en_code(v["catv"], manquants=(-1, 0))      # 0 = indéterminable
v["categorie_vehicule"] = libelle(v["catv"], CATV_GROUPE, "catv")

u = usagers.copy()
for col in ["catu", "grav", "sexe"]:
    u[col] = en_code(u[col], manquants=(-1,))
u["An_nais"] = en_code(u["An_nais"], manquants=(-1,))
u.loc[(u["An_nais"] < 1900) | (u["An_nais"] > ANNEE) | ((ANNEE - u["An_nais"]) > 105), "An_nais"] = pd.NA
u["gravite"] = libelle(u["grav"], GRAV, "grav")
u["categorie_usager"] = libelle(u["catu"], CATU, "catu")
u["sexe_libelle"] = libelle(u["sexe"], SEXE, "sexe")
display(u["gravite"].value_counts(dropna=False))

gravite
Indemne               52920
Blessé léger          49709
Blessé hospitalisé    19126
Tué                    3432
Name: count, dtype: int64

## 4. Créer les variables dérivées
- **Accident :** jour de la semaine (numéro pour le tri, lundi = 1), période de la journée, nom du mois.
- **Usager :** âge au moment de l'accident, tranche d'âge, et des indicateurs 0/1 (`est_tue`, `est_hospitalise`, `est_blesse_leger`, `est_grave`). Un usager est **grave** s'il est tué ou hospitalisé.
- **Véhicule :** catégorie regroupée (vélo, trottinette, deux-roues motorisé, voiture, poids lourd, etc.), déjà créée en 3.3.

Les indicateurs 0/1 servent dans Tableau : `SUM([est_grave]) / COUNT([id_usager])` donne directement une part.

In [14]:
JOURS = ["Lundi", "Mardi", "Mercredi", "Jeudi", "Vendredi", "Samedi", "Dimanche"]
MOIS = ["Janvier", "Février", "Mars", "Avril", "Mai", "Juin", "Juillet", "Août", "Septembre",
        "Octobre", "Novembre", "Décembre"]
c["num_jour_semaine"] = (c["date"].dt.dayofweek + 1).astype("Int64")
c["jour_semaine"] = c["date"].dt.dayofweek.map(dict(enumerate(JOURS)))
c["mois_nom"] = c["mois"].map(dict(enumerate(MOIS, start=1)))
c["periode_journee"] = pd.cut(c["heure"].astype("float"), bins=[-1, 5, 11, 17, 23],
                              labels=["Nuit (0h-6h)", "Matin (6h-12h)", "Après-midi (12h-18h)", "Soirée (18h-24h)"])

u["age"] = (ANNEE - u["An_nais"]).astype("Int64")
u["tranche_age"] = pd.cut(u["age"].astype("float"), bins=[-1, 13, 17, 24, 34, 44, 54, 64, 74, 200],
                          labels=["0-13", "14-17", "18-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75 et +"])
u["est_tue"] = (u["grav"] == 2).fillna(False).astype(int)
u["est_hospitalise"] = (u["grav"] == 3).fillna(False).astype(int)
u["est_blesse_leger"] = (u["grav"] == 4).fillna(False).astype(int)
u["est_indemne"] = (u["grav"] == 1).fillna(False).astype(int)
u["est_grave"] = u["est_tue"] | u["est_hospitalise"]
display(c[["date", "jour_semaine", "heure", "periode_journee"]].head())
display(u[["An_nais", "age", "tranche_age", "gravite", "est_grave"]].head())

NameError: name 'c' is not defined

## 5. Joindre en maîtrisant le grain
Le paramètre `validate` de `merge` **lève une erreur** si la relation entre les tables n'est pas celle qu'on annonce. Après chaque jointure, on vérifie que le nombre de lignes n'a pas augmenté.

| Jointure | Relation attendue |
|---|---|
| caractéristiques + lieux | 1 accident ↔ 1 lieu (`one_to_one`) |
| usagers + véhicules | n usagers → 1 véhicule (`many_to_one`) |
| (usagers + véhicules) + accident | n usagers → 1 accident (`many_to_one`) |

In [17]:
COLS_ACC = ["Num_Acc", "date", "date_heure", "mois", "mois_nom", "num_jour_semaine", "jour_semaine", "heure",
            "periode_journee", "zone", "dep", "com", "agglomeration", "luminosite", "luminosite_groupe",
            "intersection", "meteo", "meteo_groupe", "type_collision", "lat", "long", "coord_valides"]
COLS_LIEU = ["Num_Acc", "categorie_route", "vma", "classe_vitesse", "regime_circulation", "nbv", "profil_route",
             "trace_route", "trace_groupe", "etat_surface", "surface_groupe", "infrastructure"]

acc_lieu = c[COLS_ACC].merge(l[COLS_LIEU], on="Num_Acc", how="left", validate="one_to_one")
assert len(acc_lieu) == len(c), "La jointure caractéristiques + lieux a changé le nombre de lignes"

usa_veh = u.drop(columns=["num_veh"], errors="ignore").merge(
    v[["Num_Acc", "id_vehicule", "categorie_vehicule"]], on=["Num_Acc", "id_vehicule"],
    how="left", validate="many_to_one")
assert len(usa_veh) == len(u)

usagers_enrichis = usa_veh.merge(acc_lieu, on="Num_Acc", how="left", validate="many_to_one")
assert len(usagers_enrichis) == len(u)
print(f"usagers_enrichis : {len(usagers_enrichis):,} lignes (= {len(u):,} usagers) ✔")

usagers_enrichis : 125,187 lignes (= 125,187 usagers) ✔


**Table `accidents`** : on agrège les véhicules et les usagers au niveau de l'accident, puis on les joint à `acc_lieu` (une ligne par accident).
- Un accident est **grave** s'il compte au moins un tué ou un blessé hospitalisé.
- L'**indice de gravité** utilisé par l'ONISR est le nombre de tués pour 100 accidents.

In [18]:
agg_veh = v.groupby("Num_Acc").size().rename("nb_vehicules").reset_index()
agg_usa = u.groupby("Num_Acc").agg(nb_usagers=("id_usager", "size"), nb_tues=("est_tue", "sum"),
                                   nb_hospitalises=("est_hospitalise", "sum"),
                                   nb_blesses_legers=("est_blesse_leger", "sum"),
                                   nb_indemnes=("est_indemne", "sum")).reset_index()

accidents = (acc_lieu.merge(agg_veh, on="Num_Acc", how="left", validate="one_to_one")
                     .merge(agg_usa, on="Num_Acc", how="left", validate="one_to_one"))
cols_nb = ["nb_vehicules", "nb_usagers", "nb_tues", "nb_hospitalises", "nb_blesses_legers", "nb_indemnes"]
accidents[cols_nb] = accidents[cols_nb].fillna(0).astype(int)
accidents["accident_grave"] = ((accidents["nb_tues"] + accidents["nb_hospitalises"]) > 0).astype(int)
accidents["accident_mortel"] = (accidents["nb_tues"] > 0).astype(int)
accidents["gravite_accident"] = np.select(
    [accidents["nb_tues"] > 0, accidents["nb_hospitalises"] > 0, accidents["nb_blesses_legers"] > 0],
    ["Mortel", "Grave non mortel", "Léger"], default="Sans blessé / non renseigné")
assert len(accidents) == len(c)
print(f"accidents : {len(accidents):,} lignes (= {len(c):,} accidents) ✔")
display(accidents["gravite_accident"].value_counts())

accidents : 54,402 lignes (= 54,402 accidents) ✔


gravite_accident
Léger               34744
Grave non mortel    16432
Mortel               3226
Name: count, dtype: int64

## 6. Vérifier, puis exporter
On vérifie que les **totaux sont identiques** dans les deux tables : nombre d'accidents, nombre d'usagers, tués, hospitalisés, blessés légers. On renomme ensuite les colonnes avec des **noms explicites** et on exporte en CSV UTF-8.

In [19]:
checks = {
    "Accidents : table accidents = Num_Acc distincts de CARACTERISTIQUES": (len(accidents), carac["Num_Acc"].nunique()),
    "Accidents ayant au moins un usager : accidents = usagers_enrichis (distincts)":
        ((accidents["nb_usagers"] > 0).sum(), usagers_enrichis["Num_Acc"].nunique()),
    "Usagers : somme nb_usagers = lignes usagers_enrichis": (accidents["nb_usagers"].sum(), len(usagers_enrichis)),
    "Tués : accidents = usagers_enrichis": (accidents["nb_tues"].sum(), usagers_enrichis["est_tue"].sum()),
    "Hospitalisés : accidents = usagers_enrichis": (accidents["nb_hospitalises"].sum(), usagers_enrichis["est_hospitalise"].sum()),
    "Blessés légers : accidents = usagers_enrichis": (accidents["nb_blesses_legers"].sum(), usagers_enrichis["est_blesse_leger"].sum()),
    "Véhicules : somme nb_vehicules = lignes VEHICULES": (accidents["nb_vehicules"].sum(), len(vehicules)),
}
for nom, (a, b) in checks.items():
    controle(nom, f"{int(a):,} / {int(b):,}", int(a) == int(b))

[OK] Accidents : table accidents = Num_Acc distincts de CARACTERISTIQUES : 54,402 / 54,402
[OK] Accidents ayant au moins un usager : accidents = usagers_enrichis (distincts) : 54,402 / 54,402
[OK] Usagers : somme nb_usagers = lignes usagers_enrichis : 125,187 / 125,187
[OK] Tués : accidents = usagers_enrichis : 3,432 / 3,432
[OK] Hospitalisés : accidents = usagers_enrichis : 19,126 / 19,126
[OK] Blessés légers : accidents = usagers_enrichis : 49,709 / 49,709
[OK] Véhicules : somme nb_vehicules = lignes VEHICULES : 92,678 / 92,678


In [20]:
RENOMMER = {"dep": "departement", "com": "commune", "lat": "latitude", "long": "longitude", "vma": "vitesse_max_autorisee",
            "nbv": "nb_voies", "sexe_libelle": "sexe", "An_nais": "annee_naissance"}

cols_usagers = ["Num_Acc", "id_usager", "id_vehicule", "categorie_usager", "sexe_libelle", "An_nais", "age",
                "tranche_age", "gravite", "est_tue", "est_hospitalise", "est_blesse_leger", "est_grave",
                "categorie_vehicule"] + [col for col in COLS_ACC + COLS_LIEU if col != "Num_Acc"]
usagers_export = (usagers_enrichis[cols_usagers].drop(columns=["sexe"], errors="ignore")
                  .rename(columns=RENOMMER))
accidents_export = accidents.rename(columns=RENOMMER)

usagers_export.to_csv(PROCESSED / "usagers_enrichis.csv", index=False, encoding="utf-8")
accidents_export.to_csv(PROCESSED / "accidents.csv", index=False, encoding="utf-8")
print("Exporté :", PROCESSED / "usagers_enrichis.csv", usagers_export.shape)
print("Exporté :", PROCESSED / "accidents.csv", accidents_export.shape)
print("\nColonnes accidents :", list(accidents_export.columns))

Exporté : /home/claude/road-safety-data-analysis-2024/data/processed/usagers_enrichis.csv (125187, 46)
Exporté : /home/claude/road-safety-data-analysis-2024/data/processed/accidents.csv (54402, 42)

Colonnes accidents : ['Num_Acc', 'date', 'date_heure', 'mois', 'mois_nom', 'num_jour_semaine', 'jour_semaine', 'heure', 'periode_journee', 'zone', 'departement', 'commune', 'agglomeration', 'luminosite', 'luminosite_groupe', 'intersection', 'meteo', 'meteo_groupe', 'type_collision', 'latitude', 'longitude', 'coord_valides', 'categorie_route', 'vitesse_max_autorisee', 'classe_vitesse', 'regime_circulation', 'nb_voies', 'profil_route', 'trace_route', 'trace_groupe', 'etat_surface', 'surface_groupe', 'infrastructure', 'nb_vehicules', 'nb_usagers', 'nb_tues', 'nb_hospitalises', 'nb_blesses_legers', 'nb_indemnes', 'accident_grave', 'accident_mortel', 'gravite_accident']


## 7. Tableau des contrôles qualité
Ce tableau récapitule tous les contrôles réalisés. Il est aussi exporté dans `reports/controles_qualite.csv` et `reports/controles_qualite.md`.

In [21]:
tab_controles = pd.DataFrame(controles)
tab_controles.index = range(1, len(tab_controles) + 1)
display(tab_controles)
tab_controles.to_csv(REPORTS / "controles_qualite.csv", encoding="utf-8", index_label="N°")

lignes = ["# Tableau des contrôles qualité – BAAC 2024", "",
          "| N° | Contrôle | Résultat | Statut | Commentaire / action |", "|---|---|---|---|---|"]
for i, r in tab_controles.iterrows():
    lignes.append(f"| {i} | {r['Contrôle']} | {r['Résultat']} | {r['Statut']} | {r['Commentaire / action']} |")
(REPORTS / "controles_qualite.md").write_text("\n".join(lignes), encoding="utf-8")

,Contrôle,Résultat,Statut,Commentaire / action
1,Num_Acc unique dans CARACTERISTIQUES,"0 doublon(s) sur 54,402 lignes",OK,Clé primaire de la table accident
2,Num_Acc unique dans LIEUX,"15,846 ligne(s) en double ; 15,645 accident(s)...",À signaler,Accident au croisement de plusieurs voies : on...
3,Correspondance CARACTERISTIQUES ↔ LIEUX,"0 accident(s) sans lieu, 0 lieu(x) sans accident",OK,
4,"Valeurs -1 (non renseigné), toutes tables","549,583 cellules ; détail dans reports/valeurs...",OK,Transformées en NaN à l'étape 3 ; poids indiqu...
5,"Années de naissance aberrantes (avant 1900, ap...","3 aberrante(s), 2,579 absente(s) sur 125,187 u...",À signaler,Âge mis à NaN pour ces usagers
6,"Coordonnées absentes ou nulles (0,0)","0 absentes, 0 nulles",OK,Gardées dans les comptages mais exclues de la ...
7,Coordonnées hors de France métropolitaine (dép...,3 accident(s),À signaler,Signalées ; exclues de la carte
8,id_vehicule sans aucun usager,24 véhicule(s),OK,Conforme à la documentation ONISR (usagers en ...
9,Usagers rattachés à un véhicule absent de VEHI...,0 usager(s),OK,
10,"(Num_Acc, id_vehicule) unique dans VEHICULES",0 doublon(s),OK,


2522

## 8. Chiffres clés pour la story (option C)
Ces tableaux servent à **recouper Tableau** (les totaux doivent être identiques) et à **écrire les légendes** de la story.

Pour chaque modalité, on calcule :
- **nb_accidents** et **part_du_total** : *où* les accidents se produisent ;
- **part_graves (%)** : la part des accidents avec au moins un tué ou un hospitalisé, c'est-à-dire *à quel point* ils sont graves ;
- **tues_pour_100_acc** : l'indice de gravité.

⚠️ Comme nous n'avons ni le trafic ni le nombre de kilomètres parcourus, on compare des **parts** et des **niveaux de gravité**. On ne parle pas de « risque », et une corrélation n'est pas une cause.

In [22]:
def synthese(var):
    d = accidents.copy()
    d[var] = d[var].astype("object").fillna(NR)
    t = d.groupby(var).agg(nb_accidents=("Num_Acc", "size"), nb_graves=("accident_grave", "sum"),
                           nb_tues=("nb_tues", "sum"))
    t["part_du_total (%)"] = (100 * t["nb_accidents"] / t["nb_accidents"].sum()).round(1)
    t["part_graves (%)"] = (100 * t["nb_graves"] / t["nb_accidents"]).round(1)
    t["tues_pour_100_acc"] = (100 * t["nb_tues"] / t["nb_accidents"]).round(1)
    return t.sort_values("part_graves (%)", ascending=False)

n, g, k = len(accidents), accidents["accident_grave"].sum(), accidents["nb_tues"].sum()
print(f"Accidents corporels : {n:,} | accidents graves : {g:,} ({100*g/n:.1f} %) | tués : {k:,} "
      f"| indice de gravité : {100*k/n:.1f} tués pour 100 accidents")

for var in ["agglomeration", "categorie_route", "classe_vitesse", "luminosite_groupe", "meteo_groupe",
            "surface_groupe", "trace_groupe", "type_collision", "intersection"]:
    print(f"\n### {var}")
    display(synthese(var))

Accidents corporels : 54,402 | accidents graves : 19,658 (36.1 %) | tués : 3,432 | indice de gravité : 6.3 tués pour 100 accidents

### agglomeration


,nb_accidents,nb_graves,nb_tues,part_du_total (%),part_graves (%),tues_pour_100_acc
agglomeration,,,,,,
Hors agglomération,20392,10503,2323,37.5,51.5,11.4
En agglomération,34010,9155,1109,62.5,26.9,3.3



### categorie_route


,nb_accidents,nb_graves,nb_tues,part_du_total (%),part_graves (%),tues_pour_100_acc
categorie_route,,,,,,
Autre,216,117,23,0.4,54.2,10.6
Route départementale,20658,10456,2042,38.0,50.6,9.9
Hors réseau public,56,25,4,0.1,44.6,7.1
Parc de stationnement,439,179,25,0.8,40.8,5.7
Route nationale,3531,1406,315,6.5,39.8,8.9
Route de métropole urbaine,1500,540,88,2.8,36.0,5.9
Voie communale,22910,5723,684,42.1,25.0,3.0
Autoroute,5092,1212,251,9.4,23.8,4.9



### classe_vitesse


,nb_accidents,nb_graves,nb_tues,part_du_total (%),part_graves (%),tues_pour_100_acc
classe_vitesse,,,,,,
4. 80 km/h,7176,5152,1180,13.2,71.8,16.4
7. 130 km/h,1007,509,111,1.9,50.5,11.0
3. 60-70 km/h,3966,1583,303,7.3,39.9,7.6
Non renseigné,1789,709,117,3.3,39.6,6.5
5. 90-100 km/h,5057,1939,453,9.3,38.3,9.0
2. 40-50 km/h,24110,7316,905,44.3,30.3,3.8
6. 110 km/h,1787,529,130,3.3,29.6,7.3
1. 30 km/h ou moins,9510,1921,233,17.5,20.2,2.5


... (8 outputs left out)


### type_collision


,nb_accidents,nb_graves,nb_tues,part_du_total (%),part_graves (%),tues_pour_100_acc
type_collision,,,,,,
Frontale (2 véhicules),6057,3223,717,11.1,53.2,11.8
Non renseigné,6,3,0,0.0,50.0,0.0
Sans collision,5463,2712,524,10.0,49.6,9.6
Autre collision,15968,6762,1361,29.4,42.3,8.5
Collisions multiples (3 véhicules et +),1554,603,159,2.9,38.8,10.2
Par le côté (2 véhicules),16371,4491,406,30.1,27.4,2.5
Par l'arrière (2 véhicules),7162,1576,219,13.2,22.0,3.1
En chaîne (3 véhicules et +),1821,288,46,3.3,15.8,2.5



### intersection


,nb_accidents,nb_graves,nb_tues,part_du_total (%),part_graves (%),tues_pour_100_acc
intersection,,,,,,
Passage à niveau,138,62,25,0.3,44.9,18.1
Hors intersection,34589,13800,2748,63.6,39.9,7.9
Intersection en Y,1048,378,47,1.9,36.1,4.5
Autre intersection,2454,843,109,4.5,34.4,4.4
Intersection en T,6451,1897,198,11.9,29.4,3.1
Giratoire,2717,786,74,5.0,28.9,2.7
Intersection en X,6289,1754,213,11.6,27.9,3.4
Intersection à plus de 4 branches,263,51,7,0.5,19.4,2.7
Place,453,87,11,0.8,19.2,2.4


In [23]:
# Croisement clé : en agglomération / hors agglomération × classe de vitesse (part d'accidents graves, %)
x = accidents.assign(classe_vitesse=accidents["classe_vitesse"].fillna(NR),
                     agglomeration=accidents["agglomeration"].fillna(NR))
display(pd.crosstab(x["classe_vitesse"], x["agglomeration"], values=x["accident_grave"], aggfunc="mean")
          .mul(100).round(1))
display(pd.crosstab(x["classe_vitesse"], x["agglomeration"]))

agglomeration,En agglomération,Hors agglomération
classe_vitesse,,
1. 30 km/h ou moins,19.7,36.7
2. 40-50 km/h,29.5,39.8
3. 60-70 km/h,27.6,45.0
4. 80 km/h,61.0,72.1
5. 90-100 km/h,25.6,38.8
6. 110 km/h,27.3,29.6
7. 130 km/h,0.0,50.6
Non renseigné,29.4,60.7


agglomeration,En agglomération,Hors agglomération
classe_vitesse,,
1. 30 km/h ou moins,9210,300
2. 40-50 km/h,22030,2080
3. 60-70 km/h,1153,2813
4. 80 km/h,218,6958
5. 90-100 km/h,172,4885
6. 110 km/h,22,1765
7. 130 km/h,1,1006
Non renseigné,1204,585
